# Feature Engineering for ImpactMeter
Create `strike_rate`, `phase`, `pressure_index`, `batter_impact_score`, and `bowler_impact_score` from ball-by-ball data.

## Step 1: Load Processed Dataset

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# Works whether the notebook is run from project root or notebooks/
BASE_DIR = Path.cwd()
if not (BASE_DIR / 'data').exists():
    BASE_DIR = BASE_DIR.parent

INPUT_CSV = BASE_DIR / 'data' / 'processed' / 'ball_by_ball.csv'
OUTPUT_CSV = BASE_DIR / 'data' / 'features' / 'impact_dataset.csv'

df = pd.read_csv(INPUT_CSV)
print('Base dir:', BASE_DIR)
print('Input shape:', df.shape)
df.head(3)

Base dir: d:\VS_CODES\Projects\ImpactMeter
Input shape: (278205, 45)


C:\Users\ASUS\AppData\Local\Temp\ipykernel_13440\1663978271.py:13: DtypeWarning: Columns (0: season) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(INPUT_CSV)


,match_id,match_date,season,city,venue,match_type,gender,event_name,event_match_number,team1,...,is_wicket,wickets_on_ball,player_out,dismissal_kind,fielders,innings_runs_before,innings_wkts_before,innings_runs_after,innings_wkts_after,target_runs
0,1082591,2017-04-05,2017,Hyderabad,"Rajiv Gandhi International Stadium, Uppal",T20,male,Indian Premier League,1.0,Sunrisers Hyderabad,...,0,0,NaN,NaN,NaN,0,0,0,0,NaN
1,1082591,2017-04-05,2017,Hyderabad,"Rajiv Gandhi International Stadium, Uppal",T20,male,Indian Premier League,1.0,Sunrisers Hyderabad,...,0,0,NaN,NaN,NaN,0,0,0,0,NaN
2,1082591,2017-04-05,2017,Hyderabad,"Rajiv Gandhi International Stadium, Uppal",T20,male,Indian Premier League,1.0,Sunrisers Hyderabad,...,0,0,NaN,NaN,NaN,0,0,4,0,NaN


## Step 2: Add Core Context Features
Compute phase labels and a delivery-level pressure index.

In [2]:
def phase_from_over(over_value):
    if over_value <= 5:
        return 'powerplay'
    if over_value <= 14:
        return 'middle'
    return 'death'

df['phase'] = df['over'].apply(phase_from_over)

balls_bowled_so_far = (df['over'] * 6) + df['legal_delivery_number'].clip(lower=1)
balls_remaining = (20 * 6 - balls_bowled_so_far).clip(lower=1)

target_runs = pd.to_numeric(df['target_runs'], errors='coerce')
required_runs = (target_runs - df['innings_runs_before']).clip(lower=0)
required_rr = (required_runs * 6) / balls_remaining
rr_pressure = (required_rr / 12.0).clip(lower=0, upper=1).fillna(0)

wicket_pressure = (df['innings_wkts_before'] / 10.0).clip(lower=0, upper=1)
phase_pressure = df['phase'].map({'powerplay': 0.35, 'middle': 0.55, 'death': 0.85})

df['pressure_index'] = (0.45 * wicket_pressure + 0.35 * rr_pressure + 0.20 * phase_pressure).clip(0, 1)
max_pressure = df['pressure_index'].max()
if pd.notna(max_pressure) and max_pressure > 0:
    df['pressure_index'] = df['pressure_index'] / max_pressure

df[['phase', 'pressure_index']].head(3)

,phase,pressure_index
0,powerplay,0.075676
1,powerplay,0.075676
2,powerplay,0.075676


## Step 3: Build Batting Features
Create batter innings-level metrics including `strike_rate` and `batter_impact_score`.

In [3]:
df['is_ball_faced'] = (df['wides'] == 0).astype(int)

bat_group = ['match_id', 'match_date', 'season', 'innings', 'batting_team', 'batter']
bat_agg = df.groupby(bat_group, as_index=False).agg(
    runs_scored=('batter_runs', 'sum'),
    balls_faced=('is_ball_faced', 'sum'),
    pressure_index=('pressure_index', 'mean')
)

phase_rank = {'powerplay': 0, 'middle': 1, 'death': 2}
phase_mode = (
    df.groupby(bat_group + ['phase'], as_index=False)['is_ball_faced']
      .sum()
      .sort_values(['match_id', 'innings', 'batter', 'is_ball_faced', 'phase'], ascending=[True, True, True, False, True])
      .drop_duplicates(subset=['match_id', 'innings', 'batter'])
      .rename(columns={'phase': 'phase_bat'})
      [['match_id', 'innings', 'batter', 'phase_bat']]
)

bat_agg = bat_agg.merge(phase_mode, on=['match_id', 'innings', 'batter'], how='left')
bat_agg['phase'] = bat_agg['phase_bat'].fillna('middle')
bat_agg = bat_agg.drop(columns=['phase_bat'])

bat_agg['strike_rate'] = np.where(
    bat_agg['balls_faced'] > 0,
    (bat_agg['runs_scored'] / bat_agg['balls_faced']) * 100,
    0
)

# Context-aware but simple hackathon scoring formula; normalized around 50.
bat_agg['batter_impact_score'] = (
    50
    + 0.55 * bat_agg['runs_scored']
    + 0.12 * (bat_agg['strike_rate'] - 120)
    + 18 * (bat_agg['pressure_index'] - 0.5)
).clip(0, 100)

bat_agg.head(3)

,match_id,match_date,season,innings,batting_team,batter,runs_scored,balls_faced,pressure_index,phase,strike_rate,batter_impact_score
0,335982,2008-04-18,2007/08,1,Kolkata Knight Riders,BB McCullum,158,73,0.189330,middle,216.438356,100.000000
1,335982,2008-04-18,2007/08,1,Kolkata Knight Riders,DJ Hussey,12,12,0.248649,death,100.000000,49.675676
2,335982,2008-04-18,2007/08,1,Kolkata Knight Riders,Mohammad Hafeez,5,3,0.329730,death,166.666667,55.285135


## Step 4: Build Bowling Features
Create bowler innings-level metrics and `bowler_impact_score`.

In [4]:
df['is_legal_ball'] = df['is_legal_delivery'].astype(int)
df['runs_conceded_bowler'] = df['total_runs'] - df['byes'] - df['legbyes']

non_bowler_dismissals = {'run out', 'retired hurt', 'retired out', 'obstructing the field'}
df['bowler_wicket'] = np.where(
    (df['is_wicket'] == 1) & (~df['dismissal_kind'].str.lower().isin(non_bowler_dismissals)),
    1,
    0
)

bowl_group = ['match_id', 'match_date', 'season', 'innings', 'bowling_team', 'bowler']
bowl_agg = df.groupby(bowl_group, as_index=False).agg(
    balls_bowled=('is_legal_ball', 'sum'),
    runs_conceded=('runs_conceded_bowler', 'sum'),
    wickets_taken=('bowler_wicket', 'sum'),
    pressure_index=('pressure_index', 'mean')
)

phase_mode_bowl = (
    df.groupby(bowl_group + ['phase'], as_index=False)['is_legal_ball']
      .sum()
      .sort_values(['match_id', 'innings', 'bowler', 'is_legal_ball', 'phase'], ascending=[True, True, True, False, True])
      .drop_duplicates(subset=['match_id', 'innings', 'bowler'])
      .rename(columns={'phase': 'phase_bowl'})
      [['match_id', 'innings', 'bowler', 'phase_bowl']]
)

bowl_agg = bowl_agg.merge(phase_mode_bowl, on=['match_id', 'innings', 'bowler'], how='left')
bowl_agg['phase'] = bowl_agg['phase_bowl'].fillna('middle')
bowl_agg = bowl_agg.drop(columns=['phase_bowl'])

bowl_agg['overs_bowled'] = bowl_agg['balls_bowled'] / 6.0
bowl_agg['economy'] = np.where(
    bowl_agg['overs_bowled'] > 0,
    bowl_agg['runs_conceded'] / bowl_agg['overs_bowled'],
    np.nan
)

bowl_agg['bowler_impact_score'] = (
    50
    + 14 * bowl_agg['wickets_taken']
    + 3.5 * (7.5 - bowl_agg['economy'].fillna(10))
    + 18 * (bowl_agg['pressure_index'] - 0.5)
).clip(0, 100)

bowl_agg.head(3)

,match_id,match_date,season,innings,bowling_team,bowler,balls_bowled,runs_conceded,wickets_taken,pressure_index,phase,overs_bowled,economy,bowler_impact_score
0,335982,2008-04-18,2007/08,1,Royal Challengers Bangalore,AA Noffke,24,40,1,0.206054,death,4.0,10.0,49.958973
1,335982,2008-04-18,2007/08,1,Royal Challengers Bangalore,CL White,6,24,0,0.216216,middle,1.0,24.0,0.000000
2,335982,2008-04-18,2007/08,1,Royal Challengers Bangalore,JH Kallis,24,48,1,0.218162,middle,4.0,12.0,43.176919


## Step 5: Merge and Save Feature Dataset
Produce one player-match-innings table and save to `data/features/impact_dataset.csv`.

In [5]:
bat_merge = bat_agg.rename(columns={'batter': 'player', 'batting_team': 'team'})
bowl_merge = bowl_agg.rename(columns={'bowler': 'player', 'bowling_team': 'team'})

impact_df = bat_merge.merge(
    bowl_merge,
    on=['match_id', 'match_date', 'season', 'innings', 'player'],
    how='outer',
    suffixes=('_bat', '_bowl')
)

impact_df['team'] = impact_df['team_bat'].fillna(impact_df['team_bowl'])
impact_df['phase'] = impact_df['phase_bat'].fillna(impact_df['phase_bowl']).fillna('middle')
impact_df['pressure_index'] = impact_df['pressure_index_bat'].fillna(impact_df['pressure_index_bowl']).fillna(0.5)
impact_df['batter_impact_score'] = impact_df['batter_impact_score'].fillna(0)
impact_df['bowler_impact_score'] = impact_df['bowler_impact_score'].fillna(0)

for col in ['strike_rate', 'batter_impact_score', 'bowler_impact_score']:
    if col not in impact_df.columns:
        impact_df[col] = np.nan

final_cols = [
    'match_id', 'match_date', 'season', 'innings', 'player', 'team',
    'strike_rate', 'phase', 'pressure_index',
    'batter_impact_score', 'bowler_impact_score',
    'runs_scored', 'balls_faced', 'wickets_taken', 'economy'
]

for c in final_cols:
    if c not in impact_df.columns:
        impact_df[c] = np.nan

impact_df = impact_df[final_cols].sort_values(['match_id', 'innings', 'player'])
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
impact_df.to_csv(OUTPUT_CSV, index=False)

print('Feature dataset shape:', impact_df.shape)
print('Saved:', OUTPUT_CSV)
impact_df.head(10)

Feature dataset shape: (31609, 15)
Saved: d:\VS_CODES\Projects\ImpactMeter\data\features\impact_dataset.csv


,match_id,match_date,season,innings,player,team,strike_rate,phase,pressure_index,batter_impact_score,bowler_impact_score,runs_scored,balls_faced,wickets_taken,economy
0,335982,2008-04-18,2007/08,1,AA Noffke,Royal Challengers Bangalore,NaN,death,0.206054,0.000000,49.958973,NaN,NaN,1.0,10.000000
1,335982,2008-04-18,2007/08,1,BB McCullum,Kolkata Knight Riders,216.438356,middle,0.189330,100.000000,0.000000,158.0,73.0,NaN,NaN
2,335982,2008-04-18,2007/08,1,CL White,Royal Challengers Bangalore,NaN,middle,0.216216,0.000000,0.000000,NaN,NaN,0.0,24.000000
3,335982,2008-04-18,2007/08,1,DJ Hussey,Kolkata Knight Riders,100.000000,death,0.248649,49.675676,0.000000,12.0,12.0,NaN,NaN
4,335982,2008-04-18,2007/08,1,JH Kallis,Royal Challengers Bangalore,NaN,middle,0.218162,0.000000,43.176919,NaN,NaN,1.0,12.000000
5,335982,2008-04-18,2007/08,1,Mohammad Hafeez,Kolkata Knight Riders,166.666667,death,0.329730,55.285135,0.000000,5.0,3.0,NaN,NaN
6,335982,2008-04-18,2007/08,1,P Kumar,Royal Challengers Bangalore,NaN,powerplay,0.136649,0.000000,36.459676,NaN,NaN,0.0,9.500000
7,335982,2008-04-18,2007/08,1,RT Ponting,Kolkata Knight Riders,100.000000,middle,0.158919,52.460541,0.000000,20.0,20.0,NaN,NaN
8,335982,2008-04-18,2007/08,1,SB Joshi,Royal Challengers Bangalore,NaN,middle,0.183784,0.000000,40.224775,NaN,NaN,0.0,8.666667
9,335982,2008-04-18,2007/08,1,SC Ganguly,Kolkata Knight Riders,83.333333,powerplay,0.075676,43.462162,0.000000,10.0,12.0,NaN,NaN


## Step 6: Quick Sanity Stats

In [6]:
print('Unique players:', impact_df['player'].nunique())
print('Rows:', len(impact_df))
print('Avg batter impact:', round(impact_df['batter_impact_score'].mean(skipna=True), 2))
print('Avg bowler impact:', round(impact_df['bowler_impact_score'].mean(skipna=True), 2))

Unique players: 767
Rows: 31609
Avg batter impact: 32.94
Avg bowler impact: 24.86
